In [76]:
# ─────────────────────────────────────────────────────────────────────────────
# Jefferson Township – Monthly Panel Builder
# Inputs (clean):
#   - nh_data_clean.csv
#   - fire_and_ems_runs_clean.csv
#   - parcels_jefferson_monthly_full.csv
#
# Output:
#   - panel_monthly_with_parcels.csv  (runs + NH beds + parcel aggregates)
#
# Notes:
#   * Works at MONTHLY grain, so partial years (2018/2025) are naturally handled.
#   * Includes optional imputation for Industrial area if zeros/missing are pervasive.
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd, numpy as np
from pathlib import Path

# ========== 0) Config / Paths =================================================
ROOT      = Path().resolve().parents[0]
RAW_DIR   = ROOT / "data" / "raw"
CLEAN_DIR = ROOT / "data" / "clean"

NH_PATH    = CLEAN_DIR / "nh_data_clean.csv"
RUNS_PATH  = CLEAN_DIR / "fire_and_ems_runs_clean.csv"
PARCELS_PATH = CLEAN_DIR / "parcels_jefferson_monthly_full.csv"

OUT_PATH   = CLEAN_DIR / "panel_monthly_with_parcels.csv"

# Behavior flags
IMPUTE_INDUSTRIAL_AREA = True     # turn on/off industrial area imputation
IMPUTE_MIN_SHARE = 10             # minimum number of rows to compute per-$ sqft ratios

# ========== 1) Helper functions ==============================================

def monthify(dt_series):
    """Convert any datetime series to month-start timestamps."""
    s = pd.to_datetime(dt_series, errors="coerce")
    return s.dt.to_period("M").dt.to_timestamp()

def enforce_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def safe_ratio(n, d):
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.where(d == 0, np.nan, n / d)
    return r

# Auditor category map for parcels pclass
PCLASS_MAP = {
    "R": "1 Residential",
    "A": "2 Agricultural",
    "M": "3 Mineral",
    "C": "4 Commercial",
    "I": "5 Industrial",
    "U": "6 Public Utility Real",
    "P": "7 Public Utility Personal",
    "E": "8 General Personal",
    "Z": "8 General Personal"
}

# NFIRS-ish to Auditor categories for runs
RUNS_CAT_MAP = {
    "4 - Residential": "1 Residential",
    "3 - Health Care, Detention & Correction": "4 Commercial - Healthcare",
    "5 - Mercantile, Business": "4 Commercial",
    "1 - Assembly": "4 Commercial",
    "2 - Educational": "4 Commercial",
    "7 - Manufacturing, Processing": "5 Industrial",
    "6 - Industrial, Utility, Defense, Agriculture, Mining": "5 Industrial",
    "8 - Storage": "5 Industrial",
    "9 - Outside or Special Property": "8 General Personal",
    "10 - Common Values": "8 General Personal"
}

RUNS_RENAME = {
    "1 Residential": "runs_residential",
    "4 Commercial": "runs_commercial",
    "4 Commercial - Healthcare": "runs_healthcare",
    "5 Industrial": "runs_industrial",
    "8 General Personal": "runs_general_personal",
}

# ========== 2) Nursing homes → monthly beds ==================================
nh = pd.read_csv(NH_PATH, low_memory=False)
need = {"cms_certification_number", "number_of_certified_beds", "report_month"}
missing = need - set(nh.columns)
if missing:
    raise ValueError(f"NH input missing columns: {missing}")

nh["cms_certification_number"]   = nh["cms_certification_number"].astype(str)
nh["report_month"]               = monthify(nh["report_month"])
nh["number_of_certified_beds"]   = pd.to_numeric(nh["number_of_certified_beds"], errors="coerce")
nh = nh.dropna(subset=["report_month", "number_of_certified_beds"])

# per-facility-month (dedupe by taking max; swap to mean if preferred)
fac_m = (nh.groupby(["cms_certification_number","report_month"], as_index=False)
           .agg(beds=("number_of_certified_beds","max")))

nh_monthly = (fac_m.groupby("report_month", as_index=False)["beds"].sum()
                  .rename(columns={"beds":"nh_total_certified_beds"})
                  .sort_values("report_month"))

# complete monthly index & forward-fill (bed totals tend to be stepwise)
if not nh_monthly.empty:
    full = pd.date_range(nh_monthly["report_month"].min(),
                         nh_monthly["report_month"].max(),
                         freq="MS")
    nh_monthly = (nh_monthly.set_index("report_month")
                              .reindex(full)
                              .rename_axis("month")
                              .reset_index()
                              .rename(columns={"index":"month"}))
    nh_monthly["nh_total_certified_beds"] = nh_monthly["nh_total_certified_beds"].ffill()
else:
    nh_monthly = pd.DataFrame(columns=["month","nh_total_certified_beds"])

# ========== 3) Runs → monthly totals & by category ===========================
runs = pd.read_csv(RUNS_PATH, low_memory=False)
need = {"incident_date","incident_number","property_use_category"}
missing = need - set(runs.columns)
if missing:
    raise ValueError(f"Runs input missing columns: {missing}")

runs["month"] = monthify(runs["incident_date"])
runs = runs.dropna(subset=["month"])

# map categories
runs["auditor_category"] = runs["property_use_category"].map(RUNS_CAT_MAP).fillna("8 General Personal")

monthly_total = (runs.groupby("month", as_index=False)
                      .agg(total_calls=("incident_number","count")))

monthly_by_cat = (runs.groupby(["month","auditor_category"])["incident_number"]
                      .count()
                      .rename("runs")
                      .reset_index()
                      .pivot(index="month", columns="auditor_category", values="runs")
                      .fillna(0)
                      .reset_index())

# rename & ensure columns exist
monthly_by_cat = monthly_by_cat.rename(columns=RUNS_RENAME)
for col in RUNS_RENAME.values():
    if col not in monthly_by_cat.columns:
        monthly_by_cat[col] = 0

runs_monthly = (monthly_total.merge(monthly_by_cat, on="month", how="left")
                          .sort_values("month")
                          .reset_index(drop=True))

# ========== 4) Parcels (monthly snapshots) ===================================
parcels = pd.read_csv(PARCELS_PATH, low_memory=False)

# dates
date_col = "snapshot_month" if "snapshot_month" in parcels.columns else "report_month"
parcels["month"] = monthify(parcels[date_col])

# coercions
num_cols = ["apprlnd","apprbld","apprtot","area_a","rooms","baths","hbaths","bedrms",
            "nostory","yearblt","acrea","land_sqft"]
parcels = enforce_numeric(parcels, num_cols)

# land sqft convenience
if "land_sqft" in parcels.columns:
    parcels["land_sqft_use"] = parcels["land_sqft"]
elif "acrea" in parcels.columns:
    parcels["land_sqft_use"] = parcels["acrea"] * 43560
else:
    parcels["land_sqft_use"] = np.nan

# map pclass → auditor_category
parcels["pclass"] = parcels["pclass"].astype(str).str.upper()
parcels["auditor_category"] = parcels["pclass"].map(PCLASS_MAP).fillna("8 General Personal")

# area flags
parcels["has_area_a"]   = ~parcels["area_a"].isna()
parcels["area_a_zero"]  = (parcels["area_a"].fillna(0) == 0)
parcels["parcel_count"] = 1

# OPTIONAL: Industrial area imputation at the *row* level before aggregating
# Strategy:
#   1) Compute sqft-per-$ using categories with reliable nonzero area & apprbld/apprtot.
#   2) For Industrial rows with (area_a is NaN or 0), impute:
#        area_a_est = apprbld * median_sqft_per_$ (from Commercial first, else Residential),
#      fallback to land_sqft_use if $ info missing.
if IMPUTE_INDUSTRIAL_AREA:
    src = parcels.copy()

    # pick comparable categories: Commercial, Residential
    comp = src[src["auditor_category"].isin(["4 Commercial","1 Residential"])].copy()
    comp = comp[(comp["area_a"].fillna(0) > 0) & (comp["apprbld"].fillna(0) > 0)]

    if len(comp) >= IMPUTE_MIN_SHARE:
        comp["sqft_per_dollar"] = comp["area_a"] / comp["apprbld"]
        # robust medians by category
        med_by_cat = (comp.groupby("auditor_category", as_index=False)["sqft_per_dollar"]
                          .median()
                          .sort_values("sqft_per_dollar", ascending=False))
        # Choose primary source order: Commercial then Residential
        if "4 Commercial" in med_by_cat["auditor_category"].values:
            sqft_per_dollar = float(med_by_cat.loc[med_by_cat["auditor_category"]=="4 Commercial","sqft_per_dollar"].iloc[0])
        elif "1 Residential" in med_by_cat["auditor_category"].values:
            sqft_per_dollar = float(med_by_cat.loc[med_by_cat["auditor_category"]=="1 Residential","sqft_per_dollar"].iloc[0])
        else:
            sqft_per_dollar = np.nan
    else:
        sqft_per_dollar = np.nan

    parcels["area_a_imputed"] = np.nan
    mask_ind = parcels["auditor_category"].eq("5 Industrial") & (parcels["area_a"].fillna(0) == 0)

    # Impute from appraisal if possible
    if not np.isnan(sqft_per_dollar):
        can_use_dollar = mask_ind & parcels["apprbld"].notna() & (parcels["apprbld"] > 0)
        parcels.loc[can_use_dollar, "area_a_imputed"] = parcels.loc[can_use_dollar, "apprbld"] * sqft_per_dollar

    # Fallback to land sqft when $ is not available
    fallback = mask_ind & parcels["land_sqft_use"].notna() & parcels["area_a_imputed"].isna()
    parcels.loc[fallback, "area_a_imputed"] = parcels.loc[fallback, "land_sqft_use"]

    # Final: choose imputed where original missing/zero
    parcels["area_a_used"] = np.where(mask_ind & parcels["area_a_imputed"].notna(),
                                      parcels["area_a_imputed"],
                                      parcels["area_a"])
else:
    parcels["area_a_used"] = parcels["area_a"]

# ========== 5) Monthly parcel aggregates by category =========================
sum_vars  = [c for c in ["apprlnd","apprbld","apprtot","area_a_used","land_sqft_use"] if c in parcels.columns]
mean_vars = [c for c in ["rooms","baths","hbaths","bedrms","nostory","yearblt"] if c in parcels.columns]

agg_dict = {v:"sum" for v in sum_vars}
agg_dict.update({v:"mean" for v in mean_vars})
agg_dict.update({"parcel_count":"sum", "has_area_a":"mean"})

parcel_m = (parcels.groupby(["month","auditor_category"], as_index=False)
                   .agg(agg_dict)
                   .rename(columns={"has_area_a":"share_with_area_a"}))

def pivot_metric(df, value, suffix):
    return (df.pivot(index="month", columns="auditor_category", values=value)
              .add_suffix(f"_{suffix}")
              .reset_index())

frames = []
for col in sum_vars + mean_vars + ["parcel_count","share_with_area_a"]:
    frames.append(pivot_metric(parcel_m[["month","auditor_category", col]], col, col))

parcel_monthly_wide = frames[0]
for f in frames[1:]:
    parcel_monthly_wide = parcel_monthly_wide.merge(f, on="month", how="outer")

# Friendly renames for a few high-usage cols
parcel_monthly_wide = parcel_monthly_wide.rename(columns={
    "1 Residential_area_a_used": "res_area_sqft",
    "4 Commercial_area_a_used":  "com_area_sqft",
    "5 Industrial_area_a_used":  "ind_area_sqft",
    "8 General Personal_area_a_used": "gen_area_sqft",
    "1 Residential_apprtot": "res_apprtot",
    "4 Commercial_apprtot":  "com_apprtot",
    "5 Industrial_apprtot":  "ind_apprtot",
})

# ========== 6) Build final MONTHLY panel =====================================
panel = (runs_monthly
         .merge(nh_monthly, on="month", how="left")
         .merge(parcel_monthly_wide, on="month", how="left")
         .sort_values("month")
         .reset_index(drop=True))

# Add simple data-quality flags
panel["observed_month_flag"] = True
for c in [col for col in panel.columns if col.endswith("share_with_area_a")]:
    panel[c] = panel[c].clip(0, 1)

# ========== 6b) Feature engineering for modeling =============================
panel = panel.sort_values("month").reset_index(drop=True)

# Friendly apprbld columns (if wide ones exist after the merge)
for raw, nice in {
    "1 Residential_apprbld": "res_apprbld",
    "4 Commercial_apprbld":  "com_apprbld",
    "5 Industrial_apprbld":  "ind_apprbld",
    "8 General Personal_apprbld": "gen_apprbld",
}.items():
    if raw in panel.columns and nice not in panel.columns:
        panel[nice] = pd.to_numeric(panel[raw], errors="coerce")

# 6‑month lags (scaled): sqft → per 1,000; appraised → per $1,000,000
sqft_cols = [c for c in ["res_area_sqft","com_area_sqft","ind_area_sqft","gen_area_sqft"] if c in panel.columns]
appr_cols = [c for c in ["res_apprbld","com_apprbld","ind_apprbld","gen_apprbld"] if c in panel.columns]

for c in sqft_cols:
    panel[f"{c}_lag6"]   = panel[c].shift(6)
    panel[f"{c}_k_lag6"] = panel[f"{c}_lag6"] / 1_000.0

for c in appr_cols:
    panel[f"{c}_lag6"]   = panel[c].shift(6)
    panel[f"{c}_M_lag6"] = panel[f"{c}_lag6"] / 1_000_000.0

# Year‑over‑year deltas (12‑month differences)
panel["calls_yoy"] = panel["total_calls"] - panel["total_calls"].shift(12)
if "nh_total_certified_beds" in panel.columns:
    panel["nh_beds_yoy"] = panel["nh_total_certified_beds"] - panel["nh_total_certified_beds"].shift(12)

for c in sqft_cols:
    panel[f"{c}_k"]     = panel[c] / 1_000.0
    panel[f"{c}_k_yoy"] = panel[f"{c}_k"] - panel[f"{c}_k"].shift(12)

for c in appr_cols:
    panel[f"{c}_M"]     = panel[c] / 1_000_000.0
    panel[f"{c}_M_yoy"] = panel[f"{c}_M"] - panel[f"{c}_M"].shift(12)

# (optional) move common modeling fields to the front
front = [
    "month","total_calls","calls_yoy",
    "nh_total_certified_beds","nh_beds_yoy",
    "res_area_sqft_k_lag6","com_area_sqft_k_lag6","ind_area_sqft_k_lag6",
    "res_apprbld_M_lag6","com_apprbld_M_lag6","ind_apprbld_M_lag6",
    "res_area_sqft_k_yoy","com_area_sqft_k_yoy","ind_area_sqft_k_yoy",
    "res_apprbld_M_yoy","com_apprbld_M_yoy","ind_apprbld_M_yoy"
]
front = [c for c in front if c in panel.columns]
panel = panel[front + [c for c in panel.columns if c not in front]]

# ========== 7) Save ==========================================================
panel.to_csv(OUT_PATH, index=False)
print(f"Saved monthly panel to: {OUT_PATH}")
print(f"Rows: {len(panel):,}  |  Columns: {len(panel.columns):,}")
panel.head(3)

Saved monthly panel to: C:\Repositories\jefferson-township-run-forecasting\data\clean\panel_monthly_with_parcels.csv
Rows: 84  |  Columns: 112


,month,total_calls,calls_yoy,nh_total_certified_beds,nh_beds_yoy,res_area_sqft_k_lag6,com_area_sqft_k_lag6,ind_area_sqft_k_lag6,res_apprbld_M_lag6,com_apprbld_M_lag6,...,res_area_sqft_k,com_area_sqft_k,ind_area_sqft_k,gen_area_sqft_k,gen_area_sqft_k_yoy,res_apprbld_M,com_apprbld_M,ind_apprbld_M,gen_apprbld_M,gen_apprbld_M_yoy
0,2018-08-01,206,NaN,50,NaN,NaN,NaN,NaN,NaN,NaN,...,12154.317,33.449,4333.913388,35.914,NaN,1041.3247,44.0497,78.0394,0.0668,NaN
1,2018-09-01,216,NaN,50,NaN,NaN,NaN,NaN,NaN,NaN,...,12154.317,33.449,4333.913388,35.914,NaN,1040.8697,44.0497,78.0394,0.0668,NaN
2,2018-10-01,205,NaN,50,NaN,NaN,NaN,NaN,NaN,NaN,...,12153.060,33.449,4333.990377,35.914,NaN,1040.8407,44.0497,78.0449,0.0668,NaN
